# Personal Finance Tracker -- MCP Week 6 Project

This notebook demonstrates the core concepts from Week 6 (Model Context Protocol) by building
a personal finance tracker.

| Layer | File | What it does |
|-------|------|--------------|
| **Domain model** | `finance.py` | `FinanceAccount` class with expense/income/budget logic |
| **Persistence** | `finance_db.py` | SQLite tables for transactions and budgets |
| **MCP Server** | `finance_server.py` | FastMCP server exposing 5 tools + 2 resources over stdio |
| **This notebook** | `finance_tracker.ipynb` | Launches the server via `MCPServerStdio` and runs an agent |

### MCP concepts demonstrated

1. **Custom MCP server** built with `FastMCP`, using `@mcp.tool()` and `@mcp.resource()`
2. **Launching a server** via `MCPServerStdio` (stdio transport)
3. **Listing tools** exposed by the server
4. **Agent** that uses the MCP server's tools to manage finances conversationally
5. **SQLite persistence** behind the server

In [ ]:
from dotenv import load_dotenv
from agents import Agent, Runner, trace
from agents.mcp import MCPServerStdio
from IPython.display import display, Markdown

load_dotenv(override=True)

## Part 1 -- Explore the domain model directly

Before touching MCP, let's verify the underlying `FinanceAccount` works.

In [ ]:
from finance import FinanceAccount

account = FinanceAccount("dave")
print(account.add_expense(12.50, "Food", "Lunch at cafe"))
print(account.add_expense(45.00, "Transport", "Uber to airport"))
print(account.add_income(3000.00, "Salary", "March paycheck"))

In [ ]:
import json

summary = json.loads(account.get_summary())
print(json.dumps(summary, indent=2))

In [ ]:
print(account.set_budget("Food", 300.00))

budget = json.loads(account.budget_status())
print(json.dumps(budget, indent=2))

## Part 2 -- Launch the MCP Server, create the Agent

`finance_server.py` is a FastMCP server that exposes the same logic as **tools** and **resources**
over the stdio transport. We launch it once via `MCPServerStdio` and keep it alive across cells
using an `AsyncExitStack`. The agent is also created once and reused.

In [ ]:
from contextlib import AsyncExitStack

stack = AsyncExitStack()
finance_params = {"command": "uv", "args": ["run", "finance_server.py"]}
mcp_server = await stack.enter_async_context(
    MCPServerStdio(params=finance_params, client_session_timeout_seconds=30)
)

tools = await mcp_server.list_tools()
print(f"{len(tools)} tools available:\n")
for tool in tools:
    print(f"  {tool.name}: {tool.description[:80]}")

## Part 3 -- Agent conversations

Now we wire the running MCP server into an OpenAI Agents SDK `Agent`.
The agent is created once and reused for every request below.

In [ ]:
instructions = """You are a personal finance assistant. You help users track expenses, income, and budgets.
Always confirm what you did after each action. When showing summaries, format them clearly with markdown."""

model = "gpt-4.1-mini"

agent = Agent(name="finance_assistant", instructions=instructions, model=model, mcp_servers=[mcp_server])
print(f"Agent '{agent.name}' ready with {len(tools)} MCP tools")

In [ ]:
request = "My name is Dave. Add an expense of $25.00 in Entertainment for 'Movie tickets', then show me my summary."

with trace("add_and_summarize"):
    result = await Runner.run(agent, request)
display(Markdown(result.final_output))

In [ ]:
request = "My name is Dave. Set a budget of $200 for Shopping and $100 for Entertainment. Then show my budget status."

with trace("set_budgets"):
    result = await Runner.run(agent, request)
display(Markdown(result.final_output))

In [ ]:
request = """My name is Dave. Please do the following:

1. Add these expenses:
   - $85.00 in Housing for "Internet bill"
   - $42.50 in Food for "Grocery shopping"
   - $15.00 in Entertainment for "Spotify subscription"

2. Add income of $500.00 from Freelance for "Website project"

3. Set budgets: Food $400, Housing $1200

4. Show me a complete summary and budget status"""

with trace("full_demo"):
    result = await Runner.run(agent, request)
display(Markdown(result.final_output))

In [ ]:
await stack.aclose()
print("MCP server shut down.")